In [1]:
# ============================================================
# 0_FNS
# ============================================================

# ----------------------------######----------------------------#
#   _aiff_1001_i1_GET_id3_dict                                 #
# ----------------------------######----------------------------#

import os
from tqdm import tqdm
from pydub import AudioSegment

from mutagen.aiff import AIFF
from mutagen.mp3 import MP3
from mutagen.id3 import ID3, ID3NoHeaderError, APIC


AIFF_EXTS = (".aiff", ".aif")


def _aiff_1001_i1_GET_id3_dict(aiff_path):
    """
    Reads AIFF and returns:
      - id3_obj (mutagen ID3 container or None)
      - frames_dict: {FrameID: [stringified frame(s)]}
      - artwork: bytes or None
      - artwork_mime: mime or None
      - artwork_len: int or 0
    """
    if not os.path.isfile(aiff_path):
        return None, {}, None, None, 0

    a = AIFF(aiff_path)
    id3_obj = a.tags

    frames_dict = {}
    artwork = None
    artwork_mime = None
    artwork_len = 0

    if id3_obj:
        try:
            for fr in id3_obj.values():
                try:
                    fid = getattr(fr, "FrameID", "UNK")
                    frames_dict.setdefault(fid, []).append(str(fr))
                except Exception:
                    pass
        except Exception:
            pass

        # Try proper APIC first
        try:
            apics = id3_obj.getall("APIC")
            if apics:
                artwork = apics[0].data
                artwork_mime = apics[0].mime
        except Exception:
            pass

        # Fallback: scan any .data that looks like jpg/png
        if not artwork:
            try:
                for fr in id3_obj.values():
                    if hasattr(fr, "data") and isinstance(fr.data, (bytes, bytearray)):
                        b = bytes(fr.data)
                        if b[:3] == b"\xff\xd8\xff":
                            artwork = b
                            artwork_mime = "image/jpeg"
                            break
                        if b[:8] == b"\x89PNG\r\n\x1a\n":
                            artwork = b
                            artwork_mime = "image/png"
                            break
            except Exception:
                pass

    if artwork:
        artwork_len = len(artwork)

    return id3_obj, frames_dict, artwork, artwork_mime, artwork_len


# ----------------------------######----------------------------#
#   _mp3_1001_i1_GET_id3_dict                                  #
# ----------------------------######----------------------------#

def _mp3_1001_i1_GET_id3_dict(mp3_path):
    """
    Reads MP3 and returns:
      - id3_obj (mutagen ID3 container or None)
      - frames_dict: {FrameID: [stringified frame(s)]}
      - artwork_len: int or 0
      - comm_list: list of dicts with COMM frames (desc/lang/text)
    """
    if not os.path.isfile(mp3_path):
        return None, {}, 0, []

    try:
        m = MP3(mp3_path, ID3=ID3)
        id3_obj = m.tags
    except Exception:
        return None, {}, 0, []

    frames_dict = {}
    artwork_len = 0
    comm_list = []

    if id3_obj:
        try:
            for fr in id3_obj.values():
                try:
                    fid = getattr(fr, "FrameID", "UNK")
                    frames_dict.setdefault(fid, []).append(str(fr))
                except Exception:
                    pass
        except Exception:
            pass

        # Artwork
        try:
            apics = id3_obj.getall("APIC")
            if apics and apics[0].data:
                artwork_len = len(apics[0].data)
        except Exception:
            pass

        # COMM details (this is the one you care about)
        try:
            comms = id3_obj.getall("COMM")
            for c in comms:
                try:
                    comm_list.append({
                        "desc": getattr(c, "desc", ""),
                        "lang": getattr(c, "lang", ""),
                        "text": "\n".join([str(x) for x in getattr(c, "text", [])]) if getattr(c, "text", None) else ""
                    })
                except Exception:
                    pass
        except Exception:
            pass

    return id3_obj, frames_dict, artwork_len, comm_list


# ----------------------------######----------------------------#
#   _aiff_1001_i2_GET_cover_mime                               #
# ----------------------------######----------------------------#

def _aiff_1001_i2_GET_cover_mime(artwork_bytes, artwork_mime):
    if artwork_mime:
        return artwork_mime
    if not artwork_bytes:
        return None
    if artwork_bytes[:8] == b"\x89PNG\r\n\x1a\n":
        return "image/png"
    if artwork_bytes[:3] == b"\xff\xd8\xff":
        return "image/jpeg"
    return "image/jpeg"


# ----------------------------######----------------------------#
#   _aiff_1001_i3_GET_single_aiff_to_mp3                        #
# ----------------------------######----------------------------#

def _aiff_1001_i3_GET_single_aiff_to_mp3(aiff_path,
                                        bitrate="320k",
                                        overwrite_mp3=False):
    """
    Converts ONE AIFF to MP3:
      - Extracts AIFF ID3 frames + cover bytes
      - Converts audio to MP3
      - Deletes any existing MP3 ID3 header
      - Copies ALL AIFF ID3 frames as-is (INCLUDING COMM correctly)
      - Re-embeds cover art APIC from extracted bytes

    Returns dict with status + paths + tag info
    """
    if not os.path.isfile(aiff_path):
        return {"ok": False, "error": "AIFF not found", "aiff_path": aiff_path, "mp3_path": None}

    mp3_path = os.path.splitext(aiff_path)[0] + ".mp3"

    if os.path.isfile(mp3_path) and not overwrite_mp3:
        return {"ok": True, "skipped": True, "aiff_path": aiff_path, "mp3_path": mp3_path}

    # Extract AIFF tags + cover
    aiff_id3, aiff_frames, artwork, artwork_mime, artwork_len = _aiff_1001_i1_GET_id3_dict(aiff_path)

    # Convert audio
    audio = AudioSegment.from_file(aiff_path, format="aiff")
    audio.export(mp3_path, format="mp3", bitrate=bitrate)

    # Wipe any existing ID3 (prevents "an ID3 tag already exists")
    try:
        ID3(mp3_path).delete()
    except ID3NoHeaderError:
        pass
    except Exception:
        pass

    # Write fresh ID3
    mp3 = MP3(mp3_path, ID3=ID3)
    mp3.add_tags()

    # Copy ALL frames except APIC (we control cover separately)
    if aiff_id3:
        for fr in aiff_id3.values():
            try:
                if getattr(fr, "FrameID", None) == "APIC":
                    continue
                fr_copy = fr.copy() if hasattr(fr, "copy") else fr
                mp3.tags.add(fr_copy)
            except Exception:
                pass

    # Embed cover bytes
    if artwork:
        mime = _aiff_1001_i2_GET_cover_mime(artwork, artwork_mime)
        try:
            mp3.tags.add(
                APIC(
                    encoding=3,
                    mime=mime,
                    type=3,
                    desc="Cover",
                    data=artwork
                )
            )
        except Exception:
            pass

    mp3.save()

    # Read back for verification (COMM + art)
    _, mp3_frames, mp3_art_len, mp3_comms = _mp3_1001_i1_GET_id3_dict(mp3_path)

    return {
        "ok": True,
        "skipped": False,
        "aiff_path": aiff_path,
        "mp3_path": mp3_path,
        "aiff_art_len": artwork_len,
        "mp3_art_len": mp3_art_len,
        "aiff_has_comm": ("COMM" in aiff_frames),
        "mp3_has_comm": ("COMM" in mp3_frames),
        "mp3_comm_count": len(mp3_comms),
        "mp3_comm_preview": (mp3_comms[0]["text"][:200] if mp3_comms else "")
    }


# ----------------------------######----------------------------#
#   _aiff_1001_i4_GET_folder_aiff_to_mp3_safe                   #
# ----------------------------######----------------------------#

def _aiff_1001_i4_GET_folder_aiff_to_mp3_safe(folder_path,
                                             bitrate="320k",
                                             overwrite_mp3=False,
                                             ask_delete_aiff=True,
                                             strict_verify_comments=True,
                                             strict_verify_art=True):
    """
    MAIN: folder + subfolders
      - converts ALL AIFF/AIF -> MP3
      - preserves ALL ID3 frames from AIFF (COMM included correctly)
      - embeds cover art bytes
      - produces a strong verification report
      - asks you BEFORE deleting AIFF originals

    strict_verify_comments:
      - if True: marks file as 'verify_fail' if AIFF had COMM but MP3 doesn't,
        or if MP3 COMM count is 0 when AIFF had COMM
    strict_verify_art:
      - if True: marks verify_fail if AIFF had artwork but MP3 artwork len differs or missing
    """

    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"❌ Folder not found: {folder_path}")

    # collect
    aiff_files = []
    for root, _, files in os.walk(folder_path):
        for f in files:
            if f.lower().endswith(AIFF_EXTS) and not f.startswith("._") and f != ".DS_Store":
                aiff_files.append(os.path.join(root, f))

    if not aiff_files:
        print("⚠️ No AIFF/AIF files found.")
        return {"found": 0, "converted": 0, "skipped": 0, "failed": 0, "verify_fail": 0, "results": []}

    results = []
    converted = 0
    skipped = 0
    failed = 0
    verify_fail = 0

    for aiff_path in tqdm(aiff_files, desc="🎧 AIFF → MP3 (SAFE)"):
        try:
            r = _aiff_1001_i3_GET_single_aiff_to_mp3(
                aiff_path=aiff_path,
                bitrate=bitrate,
                overwrite_mp3=overwrite_mp3
            )

            if r.get("skipped"):
                skipped += 1
            else:
                converted += 1

            # verification flags
            vfail = False

            if strict_verify_comments:
                if r.get("aiff_has_comm") and (not r.get("mp3_has_comm") or r.get("mp3_comm_count", 0) == 0):
                    vfail = True

            if strict_verify_art:
                if (r.get("aiff_art_len", 0) > 0) and (r.get("mp3_art_len", 0) != r.get("aiff_art_len", 0)):
                    vfail = True
                if (r.get("aiff_art_len", 0) > 0) and (r.get("mp3_art_len", 0) == 0):
                    vfail = True

            r["verify_fail"] = vfail
            if vfail:
                verify_fail += 1

            results.append(r)

        except Exception as e:
            failed += 1
            results.append({"ok": False, "error": str(e), "aiff_path": aiff_path, "mp3_path": None, "verify_fail": True})

    # summary
    print("\n==================== ✅ SUMMARY ====================")
    print(f"Found AIFF:        {len(aiff_files)}")
    print(f"Converted:         {converted}")
    print(f"Skipped (exists):  {skipped}")
    print(f"Failed:            {failed}")
    print(f"VERIFY FAIL:       {verify_fail}")

    # deletion prompt
    if ask_delete_aiff:
        resp = input("\n🗑️ After you check MP3s: delete ORIGINAL AIFF files now? (y/N): ").strip().lower()
        if resp == "y":
            deleted = 0
            del_failed = 0
            # Only delete AIFFs that converted ok AND did not verify_fail
            deletable = [x["aiff_path"] for x in results if x.get("ok") and (not x.get("skipped")) and (not x.get("verify_fail"))]
            for p in tqdm(deletable, desc="🗑️ Deleting AIFF (verified-only)"):
                try:
                    os.remove(p)
                    deleted += 1
                except Exception:
                    del_failed += 1

            print(f"\n✅ Deleted AIFF (verified-only): {deleted}")
            if del_failed:
                print(f"⚠️ Could not delete: {del_failed}")
            if verify_fail:
                print("⚠️ Some files failed verification (COMM/art). Those AIFFs were NOT deleted.")
        else:
            print("ℹ️ AIFF files kept (nothing deleted).")

    return {
        "found": len(aiff_files),
        "converted": converted,
        "skipped": skipped,
        "failed": failed,
        "verify_fail": verify_fail,
        "results": results
    }


In [2]:
folder = "/Users/yerik/Music/try_new mp3"

In [3]:
res = _aiff_1001_i4_GET_folder_aiff_to_mp3_safe(
    folder_path=folder,
    bitrate="320k",
    overwrite_mp3=False,
    ask_delete_aiff=True,              # asks at end
    strict_verify_comments=True,       # FAIL if AIFF had COMM but MP3 doesn't
    strict_verify_art=True             # FAIL if cover bytes mismatch / missing
)

# If something fails, inspect:
# res["verify_fail"], res["results"]
# Example quick view of verify failures:
fails = [r for r in res["results"] if r.get("verify_fail")]
print(f"VERIFY FAIL COUNT: {len(fails)}")
for r in fails[:10]:
    print("----")
    print("AIFF:", r.get("aiff_path"))
    print("MP3 :", r.get("mp3_path"))
    print("aiff_has_comm:", r.get("aiff_has_comm"), "mp3_comm_count:", r.get("mp3_comm_count"))
    print("aiff_art_len:", r.get("aiff_art_len"), "mp3_art_len:", r.get("mp3_art_len"))
    print("comm_preview:", r.get("mp3_comm_preview"))

🎧 AIFF → MP3 (SAFE): 100%|███████████████████████████████████████████████| 1140/1140 [52:43<00:00,  2.78s/it]



==================== ✅ SUMMARY ====================
Found AIFF:        1140
Converted:         1140
Skipped (exists):  0
Failed:            0
VERIFY FAIL:       0



🗑️ After you check MP3s: delete ORIGINAL AIFF files now? (y/N):  y


🗑️ Deleting AIFF (verified-only): 100%|█████████████████████████████████| 1140/1140 [00:00<00:00, 8053.02it/s]


✅ Deleted AIFF (verified-only): 1140
VERIFY FAIL COUNT: 0


# OVERLAY POINTER FOR IMAGE COVER 

In [4]:
# ============================================================
# 0_FNS
# ============================================================

# ----------------------------######----------------------------#
# _cv_1001_overlaypng_CLEAN_GET_mp3                               #
# ----------------------------######----------------------------#

import os
import uuid
from io import BytesIO
from tqdm import tqdm

from PIL import Image
from mutagen.id3 import ID3, APIC, ID3NoHeaderError


def _cv_1001_overlaypng_CLEAN_GET_mp3(
    folder_mp3,
    overlay_png_path,
    recurse=True,

    quadrant=2,                 # 1..9 (row-major): 1 2 3 / 4 5 6 / 7 8 9
    overlay_scale=0.65,         # relative size within chosen quadrant
    anchor="center",            # center,nw,n,ne,w,e,sw,s,se
    offset_xy=(0, 0),           # (dx, dy) pixels to nudge
    max_overlay_px=None,        # optional cap on overlay max dimension
    blend_alpha=1.0,            # multiply overlay alpha (0..1)

    jpg_quality=95,             # if original cover is jpeg, re-encode jpeg at this quality
    skip_if_no_cover=True,      # if no embedded cover -> skip (True) or create blank base (False)
    blank_size=(1400, 1400),    # only used when skip_if_no_cover=False
    verbose=False
):
    """
    SAFE/CLEAN behavior:
      - Creates a temp copy of each MP3 in the same folder.
      - Writes updated cover art ONLY to the temp copy.
      - If save succeeds, atomically replaces original with temp (os.replace).
      - If anything fails, deletes temp and leaves original untouched.
      - No .bak files are created or left behind.

    Returns: list of dicts: Path, status, detail
    """

    # -------------------------
    # helpers
    # -------------------------
    def _iter_mp3_paths(root, do_recurse=True):
        if do_recurse:
            for r, _, files in os.walk(root):
                for fn in files:
                    if fn.lower().endswith(".mp3") and not fn.startswith("._") and not fn.startswith("."):
                        yield os.path.join(r, fn)
        else:
            for fn in os.listdir(root):
                p = os.path.join(root, fn)
                if os.path.isfile(p) and fn.lower().endswith(".mp3") and not fn.startswith("._") and not fn.startswith("."):
                    yield p

    def _quadrant_rect(w, h, q):
        q = int(q)
        if q < 1 or q > 9:
            raise ValueError("quadrant must be in 1..9")
        row = (q - 1) // 3
        col = (q - 1) % 3
        x0 = int(col * (w / 3))
        y0 = int(row * (h / 3))
        x1 = int((col + 1) * (w / 3))
        y1 = int((row + 1) * (h / 3))
        return x0, y0, x1, y1

    def _anchor_point(x0, y0, x1, y1, which):
        cx = (x0 + x1) // 2
        cy = (y0 + y1) // 2
        which = (which or "center").lower()
        if which == "center": return cx, cy
        if which == "nw": return x0, y0
        if which == "n":  return cx, y0
        if which == "ne": return x1, y0
        if which == "w":  return x0, cy
        if which == "e":  return x1, cy
        if which == "sw": return x0, y1
        if which == "s":  return cx, y1
        if which == "se": return x1, y1
        raise ValueError("anchor must be one of: center,nw,n,ne,w,e,sw,s,se")

    def _load_overlay_png(path):
        if not os.path.isfile(path):
            raise FileNotFoundError(f"overlay_png_path not found: {path}")
        return Image.open(path).convert("RGBA")

    def _read_cover(tags):
        apics = tags.getall("APIC")
        if not apics:
            return None
        # prefer "front cover" type=3
        for a in apics:
            if getattr(a, "type", None) == 3:
                return a
        return apics[0]

    def _img_from_apic(apic):
        return Image.open(BytesIO(apic.data)).convert("RGBA")

    def _apply_alpha_multiplier(overlay_rgba, a_mult):
        a_mult = float(a_mult)
        if a_mult >= 0.999:
            return overlay_rgba
        if a_mult <= 0:
            return Image.new("RGBA", overlay_rgba.size, (0, 0, 0, 0))
        r, g, b, a = overlay_rgba.split()
        a = a.point(lambda v: int(v * a_mult))
        return Image.merge("RGBA", (r, g, b, a))

    def _encode_image(img_rgba, prefer_mime):
        prefer_mime = (prefer_mime or "").lower()
        out = BytesIO()

        if "jpeg" in prefer_mime or "jpg" in prefer_mime:
            # flatten alpha for jpeg
            bg = Image.new("RGB", img_rgba.size, (0, 0, 0))
            bg.paste(img_rgba, mask=img_rgba.split()[-1])
            bg.save(out, format="JPEG", quality=int(jpg_quality))
            return out.getvalue(), "image/jpeg"

        img_rgba.save(out, format="PNG", optimize=True)
        return out.getvalue(), "image/png"

    def _copy_file(src, dst):
        # pure bytes copy, avoids metadata surprises
        with open(src, "rb") as fsrc, open(dst, "wb") as fdst:
            while True:
                buf = fsrc.read(1024 * 1024)
                if not buf:
                    break
                fdst.write(buf)

    # -------------------------
    # main
    # -------------------------
    if not os.path.isdir(folder_mp3):
        raise NotADirectoryError(f"folder_mp3 not found: {folder_mp3}")

    overlay_base = _load_overlay_png(overlay_png_path)
    results = []

    mp3_paths = list(_iter_mp3_paths(folder_mp3, recurse))
    if verbose:
        print(f"Found {len(mp3_paths)} mp3 files.")

    for mp3_path in tqdm(mp3_paths, desc="🎨 Cover Overlay (CLEAN)", total=len(mp3_paths)):
        tmp_path = None
        try:
            # temp file in same folder => atomic replace is safe across devices
            d = os.path.dirname(mp3_path)
            base = os.path.basename(mp3_path)
            tmp_path = os.path.join(d, f".__tmp_overlay__{uuid.uuid4().hex}__{base}")

            # work on temp copy ONLY
            _copy_file(mp3_path, tmp_path)

            # load tags from temp copy
            try:
                tags = ID3(tmp_path)
            except ID3NoHeaderError:
                if skip_if_no_cover:
                    # no ID3 at all: skip cleanly, delete temp
                    os.remove(tmp_path)
                    results.append({"Path": mp3_path, "status": "skipped_no_id3", "detail": "No ID3 header"})
                    continue
                tags = ID3()

            apic = _read_cover(tags)

            if apic is None:
                if skip_if_no_cover:
                    os.remove(tmp_path)
                    results.append({"Path": mp3_path, "status": "skipped_no_cover", "detail": "No APIC cover found"})
                    continue
                base_img = Image.new("RGBA", blank_size, (0, 0, 0, 255))
                prefer_mime = "image/png"
                apic_desc = "Cover"
                apic_type = 3
            else:
                base_img = _img_from_apic(apic)
                prefer_mime = apic.mime
                apic_desc = apic.desc
                apic_type = apic.type

            W, H = base_img.size
            x0, y0, x1, y1 = _quadrant_rect(W, H, quadrant)
            qW, qH = (x1 - x0), (y1 - y0)

            # prepare overlay
            ov = _apply_alpha_multiplier(overlay_base.copy(), blend_alpha)

            target_max_w = int(qW * float(overlay_scale))
            target_max_h = int(qH * float(overlay_scale))

            if max_overlay_px is not None:
                m = int(max_overlay_px)
                target_max_w = min(target_max_w, m)
                target_max_h = min(target_max_h, m)

            if target_max_w < 1 or target_max_h < 1:
                raise ValueError("overlay_scale/max_overlay_px made overlay too small")

            ov.thumbnail((target_max_w, target_max_h), Image.LANCZOS)
            ow, oh = ov.size

            # placement
            ax, ay = _anchor_point(x0, y0, x1, y1, anchor)
            dx, dy = offset_xy
            ax += int(dx)
            ay += int(dy)

            anch = (anchor or "center").lower()
            if anch in ("center", "n", "s"):
                px = ax - (ow // 2)
            elif anch in ("ne", "e", "se"):
                px = ax - ow
            else:
                px = ax

            if anch in ("center", "w", "e"):
                py = ay - (oh // 2)
            elif anch in ("sw", "s", "se"):
                py = ay - oh
            else:
                py = ay

            out_img = base_img.copy()
            out_img.paste(ov, (int(px), int(py)), ov)

            # re-embed cover: remove APIC only (keeps everything else)
            tags.delall("APIC")
            data_bytes, out_mime = _encode_image(out_img, prefer_mime)

            tags.add(APIC(
                encoding=3,
                mime=out_mime,
                type=apic_type if apic_type is not None else 3,
                desc=apic_desc if apic_desc else "Cover",
                data=data_bytes
            ))

            tags.save(tmp_path)

            # ATOMIC replace only after success
            os.replace(tmp_path, mp3_path)
            tmp_path = None  # so we don't try to delete it in finally

            results.append({
                "Path": mp3_path,
                "status": "ok",
                "detail": f"quadrant={quadrant}, anchor={anch}, scale={overlay_scale}, offset={offset_xy}"
            })

        except Exception as e:
            results.append({"Path": mp3_path, "status": "error", "detail": str(e)})

            # ensure temp never lingers
            try:
                if tmp_path and os.path.exists(tmp_path):
                    os.remove(tmp_path)
            except Exception:
                pass

    return results


In [5]:
# ============================================================
# !#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
# ============================================================

folder_mp3 = folder 
overlay_png_path = "/Users/yerik/Music/_1_NEW_SOURCE/_22-25_mp3/MP3.png"

results = _cv_1001_overlaypng_CLEAN_GET_mp3(
    folder_mp3=folder_mp3,
    overlay_png_path=overlay_png_path,
    recurse=True,

    quadrant=2,            # your request: top row, 2nd column
    overlay_scale=0.99,    # make smaller/larger
    anchor="center",       # move within quadrant
    offset_xy=(-79, -65),      # nudge pixels

    max_overlay_px=None,
    blend_alpha=1.0,

    skip_if_no_cover=True,
    verbose=True
)

bad = [r for r in results if r["status"] != "ok"]
print(f"OK: {sum(r['status']=='ok' for r in results)} | NOT OK: {len(bad)}")
if bad[:15]:
    print("First issues:", bad[:15])


Found 1140 mp3 files.


🎨 Cover Overlay (CLEAN): 100%|███████████████████████████████████████████| 1140/1140 [01:04<00:00, 17.78it/s]

OK: 1140 | NOT OK: 0


# ERASE BELOW

In [1]:
# # ============================================================
# # 0_FNS
# # ============================================================

# # ----------------------------######----------------------------#
# #   _aiff_0901_cover2mp3_GET_safe_folder                         #
# # ----------------------------######----------------------------#

# import os
# from tqdm import tqdm
# from pydub import AudioSegment

# from mutagen.aiff import AIFF
# from mutagen.mp3 import MP3
# from mutagen.id3 import (
#     ID3, ID3NoHeaderError,
#     APIC, TIT2, TPE1, TALB, TCON, COMM, TDRC, TKEY, TBPM
# )


# AIFF_EXTS = (".aiff", ".aif")


# def _aiff_0901_cover2mp3_GET_safe_folder(folder_path,
#                                         bitrate="320k",
#                                         overwrite_mp3=False,
#                                         ask_delete_aiff=True):
#     """
#     Recursively:
#       1) Find all AIFF/AIF in folder_path
#       2) Extract embedded cover art (APIC) + ID3 text frames from AIFF
#       3) Convert audio to MP3 (DJ-safe: 320k CBR by default)
#       4) Wipe any existing MP3 ID3 (prevents 'an ID3 tag already exists')
#       5) Write fresh tags + embed the SAME cover art bytes into MP3
#       6) Ask (at end) whether to delete original AIFFs

#     Output MP3 is created next to original AIFF with same basename.
#     """

#     if not os.path.isdir(folder_path):
#         raise FileNotFoundError(f"❌ Folder not found: {folder_path}")

#     # ---------- collect ----------
#     aiff_files = []
#     for root, _, files in os.walk(folder_path):
#         for f in files:
#             if f.lower().endswith(AIFF_EXTS) and not f.startswith("._") and f != ".DS_Store":
#                 aiff_files.append(os.path.join(root, f))

#     if not aiff_files:
#         print("⚠️ No AIFF/AIF files found.")
#         return {
#             "found": 0, "converted": 0, "skipped_exists": 0, "failed": 0,
#             "converted_paths": [], "failed_paths": []
#         }

#     converted_paths = []
#     failed_paths = []
#     skipped_exists = 0

#     # ---------- helpers ----------
#     def _aiff_extract_id3_and_artwork(aiff_path):
#         """
#         Returns:
#           aiff_id3 (mutagen ID3-like object or None),
#           artwork_bytes (bytes or None),
#           artwork_mime (str or None)
#         """
#         a = AIFF(aiff_path)

#         aiff_id3 = a.tags  # often an ID3 container when AIFF has ID3 chunk
#         artwork_bytes = None
#         artwork_mime = None

#         if aiff_id3:
#             try:
#                 apics = aiff_id3.getall("APIC")
#                 if apics:
#                     artwork_bytes = apics[0].data
#                     artwork_mime = apics[0].mime
#             except Exception:
#                 pass

#         # fallback: try to find any object with .data that looks like an image
#         if (not artwork_bytes) and aiff_id3:
#             try:
#                 for fr in aiff_id3.values():
#                     if hasattr(fr, "data") and isinstance(fr.data, (bytes, bytearray)):
#                         b = bytes(fr.data)
#                         if b[:3] == b"\xff\xd8\xff":  # JPG
#                             artwork_bytes = b
#                             artwork_mime = "image/jpeg"
#                             break
#                         if b[:8] == b"\x89PNG\r\n\x1a\n":  # PNG
#                             artwork_bytes = b
#                             artwork_mime = "image/png"
#                             break
#             except Exception:
#                 pass

#         return aiff_id3, artwork_bytes, artwork_mime

#     def _get_text(id3_obj, frame_key):
#         if not id3_obj:
#             return None
#         try:
#             frames = id3_obj.getall(frame_key)
#             if not frames:
#                 return None
#             # Most text frames store a list in .text
#             fr = frames[0]
#             if hasattr(fr, "text") and fr.text:
#                 v = str(fr.text[0]).strip()
#                 return v if v else None
#         except Exception:
#             return None
#         return None

#     # ---------- process ----------
#     for aiff_path in tqdm(aiff_files, desc="🎧 AIFF → MP3"):
#         try:
#             mp3_path = os.path.splitext(aiff_path)[0] + ".mp3"

#             if os.path.isfile(mp3_path) and not overwrite_mp3:
#                 skipped_exists += 1
#                 continue

#             # ---- extract tags + cover from AIFF ----
#             aiff_id3, artwork, artwork_mime = _aiff_extract_id3_and_artwork(aiff_path)

#             # ---- convert audio ----
#             audio = AudioSegment.from_file(aiff_path, format="aiff")
#             audio.export(mp3_path, format="mp3", bitrate=bitrate)

#             # ---- wipe ANY existing mp3 id3 header (prevents 'already exists') ----
#             try:
#                 ID3(mp3_path).delete()
#             except ID3NoHeaderError:
#                 pass
#             except Exception:
#                 # if something weird, still try to proceed with a clean add_tags step below
#                 pass

#             mp3 = MP3(mp3_path, ID3=ID3)
#             mp3.add_tags()

#             # ---- copy core text frames (from AIFF ID3 if present) ----
#             v = _get_text(aiff_id3, "TIT2")
#             if v: mp3.tags.add(TIT2(encoding=3, text=v))

#             v = _get_text(aiff_id3, "TPE1")
#             if v: mp3.tags.add(TPE1(encoding=3, text=v))

#             v = _get_text(aiff_id3, "TALB")
#             if v: mp3.tags.add(TALB(encoding=3, text=v))

#             v = _get_text(aiff_id3, "TCON")
#             if v: mp3.tags.add(TCON(encoding=3, text=v))

#             v = _get_text(aiff_id3, "TBPM")
#             if v: mp3.tags.add(TBPM(encoding=3, text=v))

#             v = _get_text(aiff_id3, "TKEY")
#             if v: mp3.tags.add(TKEY(encoding=3, text=v))

#             v = _get_text(aiff_id3, "TDRC")
#             if v: mp3.tags.add(TDRC(encoding=3, text=v))

#             # comments (keep first)
#             if aiff_id3:
#                 try:
#                     comms = aiff_id3.getall("COMM")
#                     if comms:
#                         txt = ""
#                         if hasattr(comms[0], "text") and comms[0].text:
#                             txt = str(comms[0].text[0])
#                         if txt.strip():
#                             mp3.tags.add(COMM(encoding=3, desc="Comment", text=txt.strip()))
#                 except Exception:
#                     pass

#             # ---- embed cover into MP3 ----
#             if artwork:
#                 mime = artwork_mime
#                 if not mime:
#                     if artwork[:3] == b"\xff\xd8\xff":
#                         mime = "image/jpeg"
#                     elif artwork[:8] == b"\x89PNG\r\n\x1a\n":
#                         mime = "image/png"
#                     else:
#                         mime = "image/jpeg"  # safe default

#                 mp3.tags.add(
#                     APIC(
#                         encoding=3,
#                         mime=mime,
#                         type=3,          # front cover
#                         desc="Cover",
#                         data=artwork
#                     )
#                 )

#             mp3.save()
#             converted_paths.append(aiff_path)

#         except Exception as e:
#             failed_paths.append((aiff_path, str(e)))
#             print(f"\n❌ Failed: {aiff_path}\n{e}\n")

#     # ---------- summary ----------
#     print("\n==================== ✅ SUMMARY ====================")
#     print(f"Found AIFF:        {len(aiff_files)}")
#     print(f"Converted MP3:     {len(converted_paths)}")
#     print(f"Skipped (exists):  {skipped_exists}")
#     print(f"Failed:            {len(failed_paths)}")

#     # ---------- delete prompt ----------
#     if ask_delete_aiff and converted_paths:
#         resp = input("\n🗑️ After you check MP3s: delete ORIGINAL AIFF files now? (y/N): ").strip().lower()
#         if resp == "y":
#             deleted = 0
#             delete_failed = 0
#             for p in tqdm(converted_paths, desc="🗑️ Deleting AIFF"):
#                 try:
#                     os.remove(p)
#                     deleted += 1
#                 except Exception:
#                     delete_failed += 1
#             print(f"\n✅ Deleted AIFF: {deleted}")
#             if delete_failed:
#                 print(f"⚠️ Could not delete: {delete_failed}")
#         else:
#             print("ℹ️ AIFF files kept (nothing deleted).")

#     return {
#         "found": len(aiff_files),
#         "converted": len(converted_paths),
#         "skipped_exists": skipped_exists,
#         "failed": len(failed_paths),
#         "converted_paths": converted_paths,
#         "failed_paths": failed_paths,
#     }


In [2]:
folder = "/Users/yerik/Music/_1_NEW_SOURCE/mp3"

In [3]:

res = _aiff_0901_cover2mp3_GET_safe_folder(
    folder_path=folder,
    bitrate="320k",
    overwrite_mp3=False,
    ask_delete_aiff=True
)


🎧 AIFF → MP3: 100%|██████████████████████████████████████████████████████| 1140/1140 [52:26<00:00,  2.76s/it]



==================== ✅ SUMMARY ====================
Found AIFF:        1140
Converted MP3:     1140
Skipped (exists):  0
Failed:            0



🗑️ After you check MP3s: delete ORIGINAL AIFF files now? (y/N):  y


🗑️ Deleting AIFF: 100%|█████████████████████████████████████████████████| 1140/1140 [00:00<00:00, 9079.46it/s]


✅ Deleted AIFF: 1140
